# Laboratório de Ciência de Dados
## Coleta e obtenção de dados por API: um problema de decisão com dados do IBGE


### O que você vai fazer

Nesta atividade você atuará como integrante de uma equipe de Ciência de Dados contratada para auxiliar uma empresa que pretende decidir **em qual estado brasileiro deve iniciar a expansão de um novo serviço**.

A decisão será baseada em dados oficiais do IBGE.

Você não receberá um arquivo pronto. Sua tarefa será construir a base necessária **consultando uma API pública do IBGE**, transformar as respostas em `DataFrames`, combinar os resultados e, ao final, utilizar a base construída para responder ao problema.

Nesta atividade, o foco não é aprender um algoritmo de mineração de dados. O foco é experimentar uma etapa anterior e fundamental da Ciência de Dados: **obter dados de uma fonte externa e transformá-los em uma base utilizável**.

### Objetivos

Ao final da atividade, você deverá saber:

- realizar uma requisição HTTP;
- interpretar uma resposta JSON;
- compreender uma URL de consulta a uma API;
- transformar JSON em `DataFrame`;
- modificar parâmetros de uma consulta;
- realizar várias consultas automaticamente;
- combinar resultados de diferentes consultas;
- utilizar os dados coletados para responder a um problema concreto.


# 1. O problema

Uma empresa fictícia chamada **DataMarket** pretende expandir um serviço digital para novos mercados.

A direção da empresa selecionou quatro estados para uma primeira análise:

**Minas Gerais, São Paulo, Rio de Janeiro e Espírito Santo.**

A equipe de Ciência de Dados recebeu a seguinte tarefa:

> **Construir uma base com a população estimada dos municípios desses quatro estados e determinar qual deles apresenta o maior mercado potencial.**

Para esta atividade, a empresa definiu um critério simples:

> **Mercado potencial = população total estimada do estado.**

Além disso, a equipe deverá calcular a **população média por município**, pois esse indicador ajudará a compreender como a população está distribuída entre os municípios.

Ao final, você deverá responder:

**Qual dos quatro estados apresenta o maior mercado potencial e qual apresenta a maior população média por município?**

Para responder, você precisará primeiro obter os dados.


# 2. A fonte de dados: IBGE

Utilizaremos a **API do Sistema IBGE de Recuperação Automática (SIDRA)**.

O SIDRA disponibiliza dados estatísticos do IBGE em diferentes níveis territoriais. Nesta atividade utilizaremos a **Tabela 6579 — População residente estimada**.

A consulta será feita diretamente pela API.

Uma URL de API pode conter informações que indicam:

- qual tabela será consultada;
- qual período será utilizado;
- qual nível territorial será retornado;
- qual variável será solicitada.

Não precisamos estudar toda a arquitetura de APIs. Vamos aprender apenas o suficiente para realizar as consultas necessárias.

A tabela utilizada nesta atividade é a **6579**, e a variável **9324** corresponde à população residente estimada.

Utilizaremos o ano de **2025**, cuja referência das estimativas populacionais municipais é 1º de julho de 2025.


## 2.1 Importando as bibliotecas

Utilizaremos somente `requests` e `pandas`.

A biblioteca `requests` será utilizada para fazer as requisições à API.

A biblioteca `pandas` será utilizada para organizar os dados recebidos em `DataFrames`.


In [ ]:
import requests
import pandas as pd


# 3. Primeira consulta: Minas Gerais

Vamos começar com apenas um estado.

O código da Unidade da Federação de Minas Gerais é **31**.

A consulta abaixo solicita à API:

- tabela `6579`;
- nível municipal (`n6`);
- municípios pertencentes a Minas Gerais (`in n3 31`);
- ano `2025`;
- variável `9324`.

Execute a célula exatamente como está.


In [ ]:
url = "https://apisidra.ibge.gov.br/values/t/6579/n6/in%20n3%2031/p/2025/v/9324?formato=json"

resposta = requests.get(url, timeout=30)

print("Código de status:", resposta.status_code)


Se o código apresentado for **200**, a requisição foi realizada com sucesso.

Agora vamos transformar a resposta em JSON.


In [ ]:
dados = resposta.json()

print("Tipo do objeto recebido:", type(dados))
print("Quantidade de registros:", len(dados))


# 4. Observando o JSON

A API retorna uma lista de registros.

Vamos visualizar os primeiros registros.


In [ ]:
dados[:3]


Observe que o primeiro registro possui informações sobre o nível territorial, o município, a unidade da federação, o ano e o valor da população.

Agora vamos observar os nomes dos campos disponíveis.


In [ ]:
dados[0].keys()


Um ponto importante: o JSON retornado por uma API pode conter informações que ainda não estão organizadas da maneira mais conveniente para análise.

Nossa próxima tarefa será transformar essa resposta em uma tabela.


# 5. JSON → DataFrame

O pandas consegue transformar diretamente a lista de registros em um `DataFrame`.

Execute:


In [ ]:
df_mg = pd.DataFrame(dados)

df_mg.head()


Agora observe as dimensões e as colunas.


In [ ]:
print("Linhas:", df_mg.shape[0])
print("Colunas:", df_mg.shape[1])

print("\nColunas disponíveis:")
print(df_mg.columns.tolist())


A coluna `D1N` contém o nome do município e a coluna `V` contém o valor da população.

Vamos selecionar somente as informações necessárias para o nosso problema.


In [ ]:
df_mg = df_mg[["D1N", "V"]].copy()

df_mg.columns = ["municipio", "populacao"]

df_mg.head()


# 6. Uma pequena transformação necessária

Os valores retornados pela API podem ser recebidos como texto.

Vamos verificar o tipo da coluna `populacao`.


In [ ]:
df_mg.dtypes


Para realizar cálculos numéricos, precisamos garantir que a população esteja armazenada como número.


In [ ]:
df_mg["populacao"] = pd.to_numeric(df_mg["populacao"], errors="coerce")

df_mg.dtypes


Agora podemos calcular a população total e a população média dos municípios de Minas Gerais.


In [ ]:
populacao_total_mg = df_mg["populacao"].sum()
populacao_media_mg = df_mg["populacao"].mean()

print(f"População total: {populacao_total_mg:,.0f}")
print(f"População média por município: {populacao_media_mg:,.0f}")


## Perguntas

**1. Quantos municípios de Minas Gerais foram retornados pela API?**

**2. Qual é a população total estimada de Minas Gerais na base obtida?**

**3. Qual é a população média por município?**


# 7. Entendendo a URL da API

Observe novamente a URL:

```text
https://apisidra.ibge.gov.br/values/t/6579/n6/in%20n3%2031/p/2025/v/9324?formato=json
```

Não é necessário memorizar essa estrutura.

Para esta atividade, basta entender quatro partes:

`6579` → tabela consultada

`n6` → nível municipal

`31` → código da Unidade da Federação

`2025` → ano

`9324` → variável: população residente estimada

Portanto, se quisermos consultar outro estado, precisamos alterar o código da Unidade da Federação.


# 8. Modificando a consulta

Agora vamos consultar **São Paulo**.

O código de São Paulo é **35**.

Na célula abaixo, altere apenas o código `31` para `35`.

Depois execute a consulta.


In [ ]:
url_sp = "https://apisidra.ibge.gov.br/values/t/6579/n6/in%20n3%2035/p/2025/v/9324?formato=json"

resposta_sp = requests.get(url_sp, timeout=30)

print("Código de status:", resposta_sp.status_code)

dados_sp = resposta_sp.json()

df_sp = pd.DataFrame(dados_sp)

df_sp = df_sp[["D1N", "V"]].copy()
df_sp.columns = ["municipio", "populacao"]
df_sp["populacao"] = pd.to_numeric(df_sp["populacao"], errors="coerce")

df_sp.head()


Calcule agora os dois indicadores que utilizaremos na decisão.


In [ ]:
print(f"Municípios: {len(df_sp)}")
print(f"População total: {df_sp['populacao'].sum():,.0f}")
print(f"População média por município: {df_sp['populacao'].mean():,.0f}")


# 9. Por que automatizar?

Até agora fizemos duas consultas praticamente iguais.

Imagine que a empresa tivesse solicitado dados de 10, 20 ou 27 estados. Fazer cada consulta manualmente seria trabalhoso e aumentaria a possibilidade de erros.

Em Ciência de Dados, podemos **automatizar consultas repetitivas**.

Para isso, vamos guardar os estados em um dicionário.


In [ ]:
ufs = {
    "MG": 31,
    "SP": 35,
    "RJ": 33,
    "ES": 32
}

ufs


Agora utilizaremos um `for`.

O código abaixo fará uma consulta para cada estado e guardará os resultados em uma lista.

Observe que você não precisa alterar o código.


In [ ]:
resultados = []

for sigla, codigo_uf in ufs.items():
    url = f"https://apisidra.ibge.gov.br/values/t/6579/n6/in%20n3%20{codigo_uf}/p/2025/v/9324?formato=json"

    resposta = requests.get(url, timeout=30)
    dados_uf = resposta.json()

    df_uf = pd.DataFrame(dados_uf)
    df_uf = df_uf[["D1N", "V"]].copy()
    df_uf.columns = ["municipio", "populacao"]
    df_uf["populacao"] = pd.to_numeric(df_uf["populacao"], errors="coerce")
    df_uf["uf"] = sigla

    resultados.append(df_uf)

print("Consultas realizadas:", len(resultados))


Cada elemento da lista `resultados` corresponde a um estado.

Vamos observar o primeiro deles.


In [ ]:
resultados[0].head()


# 10. Construindo uma única base

Agora vamos combinar os quatro `DataFrames` em uma única tabela.

O resultado será a base de dados que a equipe da DataMarket precisa para realizar a análise.


In [ ]:
base = pd.concat(resultados, ignore_index=True)

base.head()


In [ ]:
print("Número de registros:", len(base))
print("Número de colunas:", len(base.columns))
print("Estados presentes:", base["uf"].unique())


Confira também quantos municípios foram obtidos de cada estado.


In [ ]:
base.groupby("uf").size()


# 11. Construindo os indicadores para a decisão

Agora temos todos os dados necessários.

Vamos calcular, para cada estado:

**População total:** soma da população dos municípios.

**Número de municípios:** quantidade de municípios presentes na base.

**População média por município:** população total dividida pelo número de municípios.

Esses indicadores serão utilizados para responder ao problema inicial.


In [ ]:
resumo = (
    base
    .groupby("uf")
    .agg(
        populacao_total=("populacao", "sum"),
        numero_municipios=("municipio", "count"),
        populacao_media=("populacao", "mean")
    )
    .reset_index()
)

resumo


Vamos ordenar os estados pela população total.


In [ ]:
resumo.sort_values("populacao_total", ascending=False)


Agora vamos ordenar pela população média por município.


In [ ]:
resumo.sort_values("populacao_media", ascending=False)


# 12. O problema precisa ser respondido

Retorne ao problema apresentado no início:

> **Qual dos quatro estados apresenta o maior mercado potencial e qual apresenta a maior população média por município?**

Use a tabela `resumo` para responder.

Para facilitar a leitura, vamos apresentar os valores formatados.


In [ ]:
resultado_decisao = resumo.copy()

resultado_decisao["populacao_total"] = resultado_decisao["populacao_total"].map(lambda x: f"{x:,.0f}")
resultado_decisao["populacao_media"] = resultado_decisao["populacao_media"].map(lambda x: f"{x:,.0f}")

resultado_decisao


### Sua resposta

Escreva uma resposta curta:

**Estado com maior mercado potencial:** ____________________

**Estado com maior população média por município:** ____________________

**Explique, em duas ou três frases, o que os dois indicadores mostram sobre os quatro estados analisados.**


# 13. Indo além: qual é o município mais populoso?

A empresa também quer saber quais municípios concentram os maiores mercados individuais.

Vamos ordenar a base pela população e observar os 10 primeiros municípios.


In [ ]:
top_10 = base.sort_values("populacao", ascending=False).head(10)

top_10[["municipio", "uf", "populacao"]]


Observe que agora estamos trabalhando com os dados municipais obtidos pela API, e não apenas com os totais estaduais.

**Pergunta:** quantos dos 10 municípios mais populosos pertencem a São Paulo?


In [ ]:
top_10["uf"].value_counts()


# 14. Desafio controlado

A direção da DataMarket decidiu ampliar a análise.

Agora você deverá repetir o procedimento utilizando **Bahia (29), Paraná (41) e Santa Catarina (42)**.

A tarefa é construir uma nova base somente com esses três estados e calcular:

- população total;
- número de municípios;
- população média por município.

Não crie um novo procedimento. Adapte o código utilizado anteriormente.

Comece modificando apenas o dicionário abaixo.


Agora execute o procedimento de coleta.

Preencha o dicionário anterior antes de executar esta célula.


### Perguntas do desafio

**1. Qual dos três estados possui a maior população total?**

**2. Qual possui a maior população média por município?**

**3. Qual estado possui mais municípios?**

**4. Se o único critério da empresa fosse o tamanho do mercado potencial, qual estado deveria ser escolhido?**


# 15. O que aconteceu neste laboratório?

Você começou com um problema de decisão e não com um arquivo de dados.

Para resolver o problema, realizou o seguinte fluxo:

**Problema → Fonte de dados → API → Requisição HTTP → JSON → DataFrame → Múltiplas consultas → Base de dados → Indicadores → Decisão**

Esse fluxo é uma situação bastante comum em Ciência de Dados.

A etapa de coleta não significa simplesmente "baixar um CSV". Os dados podem estar disponíveis em serviços que precisam ser consultados programaticamente.

Nesta atividade, você utilizou uma API do IBGE para construir a própria base de análise.


# 16. Reflexão final

Responda às questões abaixo.

**1. O que é uma API?**

**2. Qual foi o papel de `requests.get()` nesta atividade?**

**3. Em que formato os dados foram inicialmente recebidos?**

**4. Por que foi necessário transformar os dados em um `DataFrame`?**

**5. Qual parte da URL foi modificada para consultar diferentes estados?**

**6. Por que utilizamos um `for` para realizar as consultas?**

**7. Qual é a vantagem de construir uma base automaticamente a partir de uma API em vez de copiar os dados manualmente?**

**8. Por que a fonte dos dados é importante em um projeto de Ciência de Dados?**


# Conclusão

Você construiu uma base de dados municipal a partir de uma fonte oficial e utilizou essa base para responder a uma questão de negócio.

O mais importante nesta atividade não foi decorar a estrutura da API. Foi compreender o papel da **obtenção programática de dados** no processo de Ciência de Dados.

Na prática, um cientista de dados frequentemente recebe um problema antes de receber os dados. Parte do trabalho consiste justamente em descobrir como obter, estruturar e organizar os dados necessários para responder à pergunta.


### Fonte dos dados

**Instituto Brasileiro de Geografia e Estatística (IBGE) — Sistema IBGE de Recuperação Automática (SIDRA).**

Tabela 6579 — População residente estimada.

A atividade utiliza a API pública do SIDRA para obter os dados diretamente do IBGE.
